# 5. Language models as judges

Two hosted models answer True, False or Uncertain for every story, with three solved examples (Appendix A). They were
queried on 8 October 2026. Hosted models change without notice, so a new run may differ.

* `gpt-6.1-sol` with low reasoning effort, on the three sets.
* `claude-opus-5-5` with the provider's default sampling, on FOLIO original and the first 100 stories with new names.

Needs `pip install -e ".[judges]"` and the keys `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` in the environment. Every
answer is saved as it arrives, so an interrupted run resumes where it stopped.

In [ ]:
# Copyright 2026 Javier Marín
# SPDX-License-Identifier: Apache-2.0

from pathlib import Path

REPO = Path.cwd().parent
DATA = REPO / "data"
RUNS = REPO / "runs"      # everything this notebook produces goes here; data/ is never overwritten

from logicverifier import analysis, data, models

sets = {name: list(stories.values()) for name, stories in data.load_test_sets(DATA / "test_sets").items()}

In [ ]:
gpt = models.ask_judge("openai", "gpt-6.1-sol", sets, RUNS / "judges" / "gpt-6.1-sol.jsonl", reasoning_effort="low")

In [ ]:
subset = {"folio_original": sets["folio_original"], "folio_new_names": sets["folio_new_names"][:100]}
claude = models.ask_judge("anthropic", "claude-opus-5-5", subset, RUNS / "judges" / "claude-opus-5-5.jsonl")

## Agreement with the answers used in the paper

In [ ]:
for model, new in (("gpt-6.1-sol", gpt), ("claude-opus-5-5", claude)):
    stored = analysis.load_judge(DATA / "judges" / f"{model}.jsonl")
    pairs = [(new[s][i], stored[s][i]) for s in new for i in new[s] if i in stored.get(s, {})]
    print(f"{model:16s} same verdict on {sum(a == b for a, b in pairs) / len(pairs):.1%} of {len(pairs)} stories")